# COVID-19 India Data Analysis & Statistical Modeling
**Analysis Window:** 30 January 2020 – 06 August 2020  
**Key Focus:** Time-series trends, state-level comparative metrics, recovery trajectory, and fatality dynamics.

## 1. Imports and Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual style settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Load cleaned dataset
df = pd.read_csv('../data/covid19_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.info()

## 2. National Overview (Latest Snapshot)

In [ ]:
summary = pd.read_csv('../data/state_summary.csv')
print('Total Confirmed:', f"{summary['Confirmed'].sum():,}")
print('Total Active:   ', f"{summary['Active'].sum():,}")
print('Total Recovered:', f"{summary['Cured'].sum():,}")
print('Total Deaths:   ', f"{summary['Deaths'].sum():,}")
summary.head(10)

## 3. Daily Trajectory and 7-Day Moving Average

In [ ]:
daily = df.groupby('Date')[['New_Cases', 'New_Recovered', 'New_Deaths']].sum().reset_index()
daily['New_Cases_7d_MA'] = daily['New_Cases'].rolling(7, min_periods=1).mean()
daily['New_Recovered_7d_MA'] = daily['New_Recovered'].rolling(7, min_periods=1).mean()

plt.figure(figsize=(12, 6))
plt.bar(daily['Date'], daily['New_Cases'], color='#4a90e2', alpha=0.35, label='Daily New Cases')
plt.plot(daily['Date'], daily['New_Cases_7d_MA'], color='#1f77b4', linewidth=2.5, label='7-Day MA (New Cases)')
plt.plot(daily['Date'], daily['New_Recovered_7d_MA'], color='#2ca02c', linewidth=2.5, linestyle='--', label='7-Day MA (Recoveries)')
plt.title('Daily COVID-19 Cases & Recoveries with 7-Day Rolling Trend', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Cases')
plt.legend()
plt.show()

## 4. Top 10 Most Affected States

In [ ]:
top10 = summary.head(10).sort_values(by='Confirmed', ascending=True)
plt.figure(figsize=(10, 6))
plt.barh(top10['State'], top10['Confirmed'], color='#e74c3c', alpha=0.85)
plt.title('Top 10 States by Confirmed COVID-19 Cases', fontsize=14, fontweight='bold')
plt.xlabel('Confirmed Cases')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.show()

## 5. Recovery Rate vs Case Fatality Rate

In [ ]:
high_conf = summary[summary['Confirmed'] >= 5000]
plt.figure(figsize=(10, 6))
scatter = plt.scatter(high_conf['Recovery_Rate_Pct'], high_conf['Mortality_Rate_Pct'],
                      s=high_conf['Confirmed'] / 1500, c=high_conf['Active'], cmap='viridis', edgecolors='black')
plt.colorbar(scatter, label='Active Cases')
for _, row in high_conf.iterrows():
    plt.annotate(row['State'], (row['Recovery_Rate_Pct'], row['Mortality_Rate_Pct']), xytext=(4, 4), textcoords='offset points', fontsize=8)
plt.title('Recovery Rate vs Fatality Rate (States >= 5,000 cases)', fontsize=13, fontweight='bold')
plt.xlabel('Recovery Rate (%)')
plt.ylabel('Case Fatality Rate (%)')
plt.show()